In [12]:
import pandas as pd

control_actions = pd.read_excel('/home/h604827/ControlActions/DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx', sheet_name='control_actions')
control_actions

,cluster_id,cluster_start,cluster_end,action_timing,action_direction,Source,Description,VT_Start,PrevValue,Value
0,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:40:59.871,100,98
1,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:41:09.258,98,96
2,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:41:10.655,96,94
3,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:41:11.558,94,92
4,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,no_change,03PIC_1013,MODE,2022-01-05 08:41:15.253,CAS,MAN
...,...,...,...,...,...,...,...,...,...,...
16089,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,before,decrease,03PIC_1013,OP,2025-06-22 17:03:51.554,72,70
16090,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,after,no_change,03FIC_3435,MODE,2025-06-22 17:19:11.717,MAN,NORMAL
16091,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,after,no_change,03PIC_1013,MODE,2025-06-22 17:19:37.402,MAN,NORMAL
16092,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,after,increase,03LIC_1071,SP,2025-06-22 17:27:43.207,40,42


## SP and OP value distributions
This section isolates SP and OP actions, normalizes signed numeric values before conversion, and creates separate summary tables for PrevValue and Value. Each table keeps OP statistics first, then SP statistics, includes the p05 to p95 common-range view, and sorts tags with available SP data to the top. If a tag has no SP actions in the sheet, its SP columns remain empty.

In [13]:
import re

import plotly.express as px
from IPython.display import display

value_cols = ['PrevValue', 'Value']
action_types = ['OP', 'SP']


def parse_action_value(value):
    if pd.isna(value):
        return float('nan')

    if isinstance(value, str):
        cleaned_value = re.sub(r'^([+-])\s+', r'\1', value.strip())
        return pd.to_numeric(cleaned_value, errors='coerce')

    return pd.to_numeric(value, errors='coerce')


tag_action_values = (
    control_actions.loc[
        control_actions['Description'].isin(action_types),
        ['cluster_id', 'Source', 'Description', 'action_timing', 'action_direction', 'VT_Start', *value_cols],
    ]
    .copy()
    .rename(columns={'Source': 'tag', 'Description': 'action_type'})
)

for col in value_cols:
    tag_action_values[col] = tag_action_values[col].apply(parse_action_value)

tag_action_values_long = (
    tag_action_values
    .melt(
        id_vars=['cluster_id', 'tag', 'action_type', 'action_timing', 'action_direction', 'VT_Start'],
        value_vars=value_cols,
        var_name='value_field',
        value_name='action_value',
    )
    .dropna(subset=['action_value'])
)

tag_action_values_long['value_field'] = pd.Categorical(
    tag_action_values_long['value_field'],
    categories=['PrevValue', 'Value'],
    ordered=True,
)

tag_order = (
    tag_action_values_long.groupby('tag')
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)

tag_action_values_long.head()

,cluster_id,tag,action_type,action_timing,action_direction,VT_Start,value_field,action_value
0,1,03HIC_1141,OP,before,decrease,2022-01-05 08:40:59.871,PrevValue,100.0
1,1,03HIC_1141,OP,before,decrease,2022-01-05 08:41:09.258,PrevValue,98.0
2,1,03HIC_1141,OP,before,decrease,2022-01-05 08:41:10.655,PrevValue,96.0
3,1,03HIC_1141,OP,before,decrease,2022-01-05 08:41:11.558,PrevValue,94.0
4,1,03HIC_1141,OP,before,decrease,2022-01-05 08:41:24.584,PrevValue,92.0


In [14]:
distribution_summary = (
    tag_action_values_long
    .groupby(['tag', 'action_type', 'value_field'], observed=True)['action_value']
    .agg(
        count='size',
        mean='mean',
        std='std',
        min='min',
        p05=lambda s: s.quantile(0.05),
        q25=lambda s: s.quantile(0.25),
        median='median',
        q75=lambda s: s.quantile(0.75),
        p95=lambda s: s.quantile(0.95),
        max='max',
    )
    .reset_index()
    .sort_values(['tag', 'value_field', 'action_type'])
)

metric_cols = ['mean', 'std', 'min', 'p05', 'q25', 'median', 'q75', 'p95', 'max']
summary_stat_cols = ['count', *metric_cols]
distribution_summary[metric_cols] = distribution_summary[metric_cols].round(3)


def format_common_range(low_value, high_value):
    if pd.isna(low_value) or pd.isna(high_value):
        return pd.NA
    return f'[{low_value:.3f}, {high_value:.3f}]'



def build_value_summary_table(value_field):
    value_field_summary = distribution_summary.loc[
        distribution_summary['value_field'] == value_field,
        ['tag', 'action_type', *summary_stat_cols],
    ].copy()

    op_summary = (
        value_field_summary.loc[value_field_summary['action_type'] == 'OP']
        .drop(columns='action_type')
        .rename(columns={col: f'{col}_OP' for col in summary_stat_cols})
    )

    sp_summary = (
        value_field_summary.loc[value_field_summary['action_type'] == 'SP']
        .drop(columns='action_type')
        .rename(columns={col: f'{col}_SP' for col in summary_stat_cols})
    )

    summary_table = op_summary.merge(sp_summary, on='tag', how='outer')

    summary_table['common_range_OP'] = summary_table.apply(
        lambda row: format_common_range(row['p05_OP'], row['p95_OP']),
        axis=1,
    )
    summary_table['common_range_SP'] = summary_table.apply(
        lambda row: format_common_range(row['p05_SP'], row['p95_SP']),
        axis=1,
    )

    summary_table = (
        summary_table
        .assign(
            has_sp=summary_table['count_SP'].notna(),
            has_op=summary_table['count_OP'].notna(),
        )
        .sort_values(
            ['has_sp', 'has_op', 'count_SP', 'count_OP', 'tag'],
            ascending=[False, False, False, False, True],
            na_position='last',
        )
        .drop(columns=['has_sp', 'has_op'])
        .reset_index(drop=True)
    )

    ordered_cols = [
        'tag',
        *[f'{col}_OP' for col in summary_stat_cols],
        'common_range_OP',
        *[f'{col}_SP' for col in summary_stat_cols],
        'common_range_SP',
    ]
    return summary_table[ordered_cols]


prev_value_summary = build_value_summary_table('PrevValue')
value_summary = build_value_summary_table('Value')

print('PrevValue summary')
display(prev_value_summary[prev_value_summary['tag'].isin(['03LIC_1071', '03LIC_1016', '03PIC_1013'])])

print('Value summary')
display(value_summary[value_summary['tag'].isin(['03LIC_1071', '03LIC_1016', '03PIC_1013'])])

PrevValue summary


,tag,count_OP,mean_OP,std_OP,min_OP,p05_OP,q25_OP,median_OP,q75_OP,p95_OP,...,mean_SP,std_SP,min_SP,p05_SP,q25_SP,median_SP,q75_SP,p95_SP,max_SP,common_range_SP
1,03LIC_1071,794.0,42.958,18.311,-6.88,2.0,34.241,49.000,55.000,64.029,...,37.180,8.063,0.000,25.000,34.0,36.566,40.0,50.0,73.271,"[25.000, 50.000]"
3,03LIC_1016,335.0,10.348,14.728,-6.88,-5.0,0.000,5.000,16.000,40.000,...,38.288,9.306,8.954,23.692,35.0,38.000,42.0,50.0,84.853,"[23.692, 50.000]"
16,03PIC_1013,1834.0,61.624,11.971,0.00,45.0,54.000,60.806,70.336,80.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


Value summary


,tag,count_OP,mean_OP,std_OP,min_OP,p05_OP,q25_OP,median_OP,q75_OP,p95_OP,...,mean_SP,std_SP,min_SP,p05_SP,q25_SP,median_SP,q75_SP,p95_SP,max_SP,common_range_SP
1,03LIC_1071,794.0,42.741,18.415,-6.88,2.000,34.098,49.000,55.0,64.000,...,37.340,6.169,10.0,28.0,35.0,38.0,40.00,45.0,60.0,"[28.000, 45.000]"
3,03LIC_1016,335.0,10.131,14.256,-6.88,-4.264,0.000,5.000,16.0,40.000,...,37.612,6.317,20.0,25.0,35.0,39.0,41.17,45.0,60.0,"[25.000, 45.000]"
16,03PIC_1013,1834.0,61.571,11.842,2.00,45.065,54.000,60.762,70.2,79.941,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


In [15]:
distribution_fig = px.violin(
    tag_action_values_long,
    x='tag',
    y='action_value',
    color='value_field',
    facet_row='action_type',
    box=True,
    points=False,
    category_orders={
        'tag': tag_order,
        'action_type': ['OP', 'SP'],
        'value_field': ['PrevValue', 'Value'],
    },
    title='Per-tag distribution of PrevValue and Value for SP and OP actions',
)

distribution_fig.update_layout(height=900, violinmode='overlay')
distribution_fig.update_xaxes(tickangle=45)

distribution_fig

In [16]:
control_actions

,cluster_id,cluster_start,cluster_end,action_timing,action_direction,Source,Description,VT_Start,PrevValue,Value
0,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:40:59.871,100,98
1,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:41:09.258,98,96
2,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:41:10.655,96,94
3,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,decrease,03HIC_1141,OP,2022-01-05 08:41:11.558,94,92
4,1,2022-01-05 08:53:41.853,2022-01-05 09:33:33.105,before,no_change,03PIC_1013,MODE,2022-01-05 08:41:15.253,CAS,MAN
...,...,...,...,...,...,...,...,...,...,...
16089,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,before,decrease,03PIC_1013,OP,2025-06-22 17:03:51.554,72,70
16090,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,after,no_change,03FIC_3435,MODE,2025-06-22 17:19:11.717,MAN,NORMAL
16091,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,after,no_change,03PIC_1013,MODE,2025-06-22 17:19:37.402,MAN,NORMAL
16092,539,2025-06-22 17:14:12.209,2025-06-22 17:14:56.205,after,increase,03LIC_1071,SP,2025-06-22 17:27:43.207,40,42


In [17]:
direction_actions = (
    control_actions.loc[
        control_actions['Description'].isin(['SP', 'OP']),
        ['Source', 'Description', 'action_direction'],
    ]
    .copy()
    .rename(columns={'Source': 'tag'})
)

direction_actions['action_direction'] = (
    direction_actions['action_direction']
    .astype(str)
    .str.strip()
    .str.lower()
)

direction_summary = (
    direction_actions.loc[
        direction_actions['action_direction'].isin(['increase', 'decrease'])
    ]
    .groupby(['tag', 'action_direction'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['increase', 'decrease'], fill_value=0)
    .rename(columns={
        'increase': 'increase_count',
        'decrease': 'decrease_count',
    })
    .reset_index()
)

direction_summary['total_actions'] = (
    direction_summary['increase_count'] + direction_summary['decrease_count']
)

direction_summary = direction_summary.sort_values(
    ['total_actions', 'increase_count', 'decrease_count', 'tag'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

direction_summary

action_direction,tag,increase_count,decrease_count,total_actions
0,03FIC_3435,1405,807,2212
1,03PIC_1013,831,1000,1831
2,03HIC_1151,735,928,1663
3,03HIC_3100,515,1098,1613
4,03LIC_1071,597,449,1046
5,03HIC_3132,864,155,1019
6,03HIC_1141,290,698,988
7,03PIC_3131,206,458,664
8,03LIC_1034,301,291,592
9,03LIC_3153,402,99,501


In [20]:
import re
import numpy as np
from IPython.display import Markdown, display

pv_data_path = '/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet'
tags_of_interest = ['03LIC_1071', '03LIC_1016', '03PIC_1013']
action_type_order = ['OP', 'SP']
pv_reference_col = '03LIC_1071.PV'
pv_clip_min = 0.0
pv_clip_max = 100.0
pv_bin_width = 5.0


def parse_step_value(value):
    if pd.isna(value):
        return float('nan')

    if isinstance(value, str):
        cleaned_value = re.sub(r'^([+-])\s+', r'\1', value.strip())
        return pd.to_numeric(cleaned_value, errors='coerce')

    return pd.to_numeric(value, errors='coerce')


minute_pv_data = pd.read_parquet(pv_data_path)

if 'TimeStamp' in minute_pv_data.columns:
    pv_lookup = minute_pv_data[['TimeStamp', pv_reference_col]].copy()
else:
    pv_lookup = minute_pv_data[[pv_reference_col]].reset_index()
    pv_lookup = pv_lookup.rename(columns={pv_lookup.columns[0]: 'TimeStamp'})

pv_lookup['TimeStamp'] = pd.to_datetime(pv_lookup['TimeStamp'], errors='coerce')
pv_lookup[pv_reference_col] = pd.to_numeric(pv_lookup[pv_reference_col], errors='coerce')
pv_lookup = (
    pv_lookup
    .dropna(subset=['TimeStamp', pv_reference_col])
    .sort_values('TimeStamp')
)

pv_range_actions = (
    control_actions.loc[
        control_actions['Source'].isin(tags_of_interest)
        & control_actions['Description'].isin(action_type_order),
        ['Source', 'Description', 'VT_Start', 'PrevValue', 'Value'],
    ]
    .copy()
    .rename(columns={'Source': 'tag', 'Description': 'action_type'})
)

pv_range_actions['VT_Start'] = pd.to_datetime(pv_range_actions['VT_Start'], errors='coerce')
pv_range_actions['prev_value_num'] = pv_range_actions['PrevValue'].apply(parse_step_value)
pv_range_actions['value_num'] = pv_range_actions['Value'].apply(parse_step_value)
pv_range_actions['step_change'] = (
    pv_range_actions['value_num'] - pv_range_actions['prev_value_num']
)
pv_range_actions = pv_range_actions.dropna(subset=['VT_Start']).sort_values('VT_Start')

actions_with_1071_pv = pd.merge_asof(
    pv_range_actions,
    pv_lookup,
    left_on='VT_Start',
    right_on='TimeStamp',
    direction='backward',
    tolerance=pd.Timedelta('1min'),
).rename(columns={pv_reference_col: 'pv_1071'})

matched_actions_with_1071_pv = actions_with_1071_pv.dropna(subset=['pv_1071']).copy()

if matched_actions_with_1071_pv.empty:
    raise ValueError('No SP or OP actions matched to 03LIC_1071.PV within one minute.')

matched_actions_with_1071_pv['pv_1071'] = matched_actions_with_1071_pv['pv_1071'].clip(
    lower=pv_clip_min,
    upper=pv_clip_max,
)

pv_bin_edges = np.arange(pv_clip_min, pv_clip_max + pv_bin_width, pv_bin_width)
pv_bin_labels = [
    f'[{left:.1f}, {right:.1f})' if right < pv_clip_max else f'[{left:.1f}, {right:.1f}]'
    for left, right in zip(pv_bin_edges[:-1], pv_bin_edges[1:])
]
pv_values_for_binning = matched_actions_with_1071_pv['pv_1071'].where(
    matched_actions_with_1071_pv['pv_1071'] < pv_clip_max,
    np.nextafter(pv_clip_max, pv_clip_min),
)

matched_actions_with_1071_pv['pv_1071_range'] = pd.cut(
    pv_values_for_binning,
    bins=pv_bin_edges,
    labels=pv_bin_labels,
    include_lowest=True,
    right=False,
)

step_change_actions = matched_actions_with_1071_pv.dropna(subset=['step_change']).copy()
step_change_actions = step_change_actions.loc[
    step_change_actions['step_change'] != 0
].copy()

if step_change_actions.empty:
    raise ValueError('No numeric non-zero step changes were found for the selected SP and OP actions.')

step_change_actions['step_direction'] = np.where(
    step_change_actions['step_change'] > 0,
    'increase',
    'decrease',
)

step_change_stats_by_1071_pv_range = (
    step_change_actions
    .groupby(['tag', 'action_type', 'pv_1071_range'], observed=True)['step_change']
    .agg(
        step_change_count='size',
        min='min',
        q25=lambda s: s.quantile(0.25),
        mean='mean',
        median='median',
        q75=lambda s: s.quantile(0.75),
        max='max',
        std='std',
    )
    .reset_index()
)

step_stat_cols = ['min', 'q25', 'mean', 'median', 'q75', 'max', 'std']
step_change_stats_by_1071_pv_range[step_stat_cols] = (
    step_change_stats_by_1071_pv_range[step_stat_cols].round(3)
)
step_change_stats_by_1071_pv_range = step_change_stats_by_1071_pv_range.sort_values(
    ['tag', 'action_type', 'pv_1071_range']
).reset_index(drop=True)

direction_by_1071_pv_range = (
    step_change_actions
    .groupby(['tag', 'action_type', 'pv_1071_range', 'step_direction'], observed=True)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['increase', 'decrease'], fill_value=0)
    .rename(columns={
        'increase': 'increase_count',
        'decrease': 'decrease_count',
    })
    .reset_index()
)

direction_by_1071_pv_range['total_actions'] = (
    direction_by_1071_pv_range['increase_count'] + direction_by_1071_pv_range['decrease_count']
)
direction_by_1071_pv_range = direction_by_1071_pv_range.sort_values(
    ['tag', 'action_type', 'pv_1071_range']
).reset_index(drop=True)

op_step_change_stats_by_1071_pv_range = step_change_stats_by_1071_pv_range.loc[
    step_change_stats_by_1071_pv_range['action_type'] == 'OP'
]
sp_step_change_stats_by_1071_pv_range = step_change_stats_by_1071_pv_range.loc[
    step_change_stats_by_1071_pv_range['action_type'] == 'SP'
]
op_direction_by_1071_pv_range = direction_by_1071_pv_range.loc[
    direction_by_1071_pv_range['action_type'] == 'OP'
]
sp_direction_by_1071_pv_range = direction_by_1071_pv_range.loc[
    direction_by_1071_pv_range['action_type'] == 'SP'
]

step_change_tables = {}
direction_tables = {}

empty_step_change_table = pd.DataFrame(
    columns=['pv_1071_range', 'step_change_count', 'min', 'q25', 'mean', 'median', 'q75', 'max', 'std']
)
empty_direction_table = pd.DataFrame(
    columns=['pv_1071_range', 'increase_count', 'decrease_count', 'total_actions']
)

print(
    f"Matched {len(matched_actions_with_1071_pv)} of {len(pv_range_actions)} SP/OP actions to 03LIC_1071.PV "
    f"using the latest minute value at or before each action time after clipping to [{pv_clip_min:.0f}, {pv_clip_max:.0f}]."
)
print(
    f"{len(step_change_actions)} actions have numeric non-zero step changes after computing Value - PrevValue."
)
print('Step change is computed as Value - PrevValue. Negative values mean the operator reduced the setting.')

for tag in tags_of_interest:
    display(Markdown(f'## {tag}'))

    for action_type in action_type_order:
        step_table = (
            step_change_stats_by_1071_pv_range.loc[
                (step_change_stats_by_1071_pv_range['tag'] == tag)
                & (step_change_stats_by_1071_pv_range['action_type'] == action_type),
                ['pv_1071_range', 'step_change_count', 'min', 'q25', 'mean', 'median', 'q75', 'max', 'std'],
            ]
            .reset_index(drop=True)
        )
        direction_table = (
            direction_by_1071_pv_range.loc[
                (direction_by_1071_pv_range['tag'] == tag)
                & (direction_by_1071_pv_range['action_type'] == action_type),
                ['pv_1071_range', 'increase_count', 'decrease_count', 'total_actions'],
            ]
            .reset_index(drop=True)
        )

        if step_table.empty:
            step_table = empty_step_change_table.copy()
        if direction_table.empty:
            direction_table = empty_direction_table.copy()

        step_change_tables[(tag, action_type)] = step_table
        direction_tables[(tag, action_type)] = direction_table

        display(Markdown(f'### {action_type} step-change stats by clipped 03LIC_1071.PV range'))
        display(step_table)

        display(Markdown(f'### {action_type} increase and decrease counts by clipped 03LIC_1071.PV range'))
        display(direction_table)

Matched 3395 of 3407 SP/OP actions to 03LIC_1071.PV using the latest minute value at or before each action time after clipping to [0, 100].
3391 actions have numeric non-zero step changes after computing Value - PrevValue.
Step change is computed as Value - PrevValue. Negative values mean the operator reduced the setting.


## 03LIC_1071

### OP step-change stats by clipped 03LIC_1071.PV range

,pv_1071_range,step_change_count,min,q25,mean,median,q75,max,std
0,"[0.0, 5.0)",177,-38.708,2.00,1.521,2.0,2.00,50.000,6.504
1,"[5.0, 10.0)",33,-5.000,1.00,2.500,2.0,2.00,25.000,5.660
2,"[10.0, 15.0)",20,-5.000,-2.00,1.300,2.0,2.00,20.000,5.391
3,"[15.0, 20.0)",19,-5.000,-2.00,1.415,1.5,2.50,11.880,4.840
4,"[20.0, 25.0)",18,-2.000,-2.00,2.556,2.0,2.00,20.000,5.649
5,"[25.0, 30.0)",39,-29.244,-0.45,-0.737,0.1,0.10,35.000,9.642
6,"[30.0, 35.0)",46,-49.791,-0.10,-0.878,-0.1,0.10,5.000,7.471
7,"[35.0, 40.0)",62,-20.000,-2.00,-0.924,-0.1,-0.10,5.000,3.729
8,"[40.0, 45.0)",48,-7.303,-2.00,0.177,-0.1,2.00,10.000,3.314
9,"[45.0, 50.0)",32,-5.000,-2.00,0.417,2.0,2.00,9.635,2.772


### OP increase and decrease counts by clipped 03LIC_1071.PV range

...

step_direction,pv_1071_range,increase_count,decrease_count,total_actions
0,"[0.0, 5.0)",53,24,77
1,"[5.0, 10.0)",5,0,5
2,"[10.0, 15.0)",7,31,38
3,"[15.0, 20.0)",7,51,58
4,"[20.0, 25.0)",14,54,68
5,"[25.0, 30.0)",27,54,81
6,"[30.0, 35.0)",81,138,219
7,"[35.0, 40.0)",118,278,396
8,"[40.0, 45.0)",122,167,289
9,"[45.0, 50.0)",101,70,171


### SP step-change stats by clipped 03LIC_1071.PV range

,pv_1071_range,step_change_count,min,q25,mean,median,q75,max,std


### SP increase and decrease counts by clipped 03LIC_1071.PV range

,pv_1071_range,increase_count,decrease_count,total_actions


In [7]:
# Alarms based on 02FI_1000 levels
import pandas as pd
from IPython.display import display

LOWER_LIMIT_FI1000 = 8.386505779
UPPER_LIMIT_FI1000 = 8.630339925

alarm_workbook_path = '/home/h604827/ControlActions/DATA/1071_pvlo_alarms_clustered_with_control_actions.xlsx'
alarm_sheet_name = 'overlapping_alarms'
fallback_pv_data_path = '/home/h604827/ControlActions/DATA/03LIC_1071_JAN_2026_filtered.parquet'
fi1000_col = '02FI_1000.PV'
expansion_schedule = [0.0, 0.10, 0.20, 0.30]


def strip_timezone(datetime_series):
    if getattr(datetime_series.dt, 'tz', None) is not None:
        return datetime_series.dt.tz_localize(None)
    return datetime_series


if 'minute_pv_data' not in globals():
    minute_pv_data = pd.read_parquet(globals().get('pv_data_path', fallback_pv_data_path))

if fi1000_col not in minute_pv_data.columns:
    raise KeyError(f'{fi1000_col} was not found in the loaded PV/OP data.')

if 'TimeStamp' in minute_pv_data.columns:
    fi1000_series = minute_pv_data[['TimeStamp', fi1000_col]].copy()
    fi1000_series['TimeStamp'] = pd.to_datetime(fi1000_series['TimeStamp'], errors='coerce')
    fi1000_series = fi1000_series.dropna(subset=['TimeStamp'])
    fi1000_series['TimeStamp'] = strip_timezone(fi1000_series['TimeStamp'])
    fi1000_series = fi1000_series.set_index('TimeStamp')[fi1000_col]
else:
    fi1000_series = minute_pv_data[[fi1000_col]].copy().reset_index()
    fi1000_series = fi1000_series.rename(columns={fi1000_series.columns[0]: 'TimeStamp'})
    fi1000_series['TimeStamp'] = pd.to_datetime(fi1000_series['TimeStamp'], errors='coerce')
    fi1000_series = fi1000_series.dropna(subset=['TimeStamp'])
    fi1000_series['TimeStamp'] = strip_timezone(fi1000_series['TimeStamp'])
    fi1000_series = fi1000_series.set_index('TimeStamp')[fi1000_col]

fi1000_series = pd.to_numeric(fi1000_series, errors='coerce').sort_index()
fi1000_series = fi1000_series[~fi1000_series.index.duplicated(keep='last')]

overlapping_alarms = pd.read_excel(alarm_workbook_path, sheet_name=alarm_sheet_name).copy()
raw_alarm_count = len(overlapping_alarms)
required_alarm_cols = ['alarm_start', 'alarm_end']
missing_alarm_cols = [col for col in required_alarm_cols if col not in overlapping_alarms.columns]
if missing_alarm_cols:
    raise KeyError(
        f'Missing required columns in {alarm_sheet_name}: {missing_alarm_cols}'
    )

for col in required_alarm_cols:
    overlapping_alarms[col] = pd.to_datetime(overlapping_alarms[col], errors='coerce')
    overlapping_alarms[col] = strip_timezone(overlapping_alarms[col])

alarm_windows = overlapping_alarms.dropna(subset=required_alarm_cols).copy()
alarm_windows = alarm_windows.loc[alarm_windows['alarm_end'] >= alarm_windows['alarm_start']].copy()
alarm_windows = alarm_windows.reset_index(drop=True)
alarm_windows['alarm_id'] = alarm_windows.index + 1
alarm_windows['sample_start'] = alarm_windows['alarm_start'].dt.ceil('min')
alarm_windows['sample_end'] = alarm_windows['alarm_end'].dt.floor('min')

base_span = UPPER_LIMIT_FI1000 - LOWER_LIMIT_FI1000


def evaluate_alarm_window(sample_start, sample_end, lower_limit, upper_limit):
    if pd.isna(sample_start) or pd.isna(sample_end) or sample_end < sample_start:
        return pd.Series(
            {
                'expected_samples': 0,
                'observed_samples': 0,
                'full_coverage': False,
                'outside_limit_count': 0,
                'min_fi1000': float('nan'),
                'max_fi1000': float('nan'),
                'within_limits': False,
            }
        )

    window = fi1000_series.loc[sample_start:sample_end]
    expected_samples = int((sample_end - sample_start) / pd.Timedelta(minutes=1)) + 1
    observed_samples = len(window)
    valid_window = window.dropna()
    full_coverage = observed_samples == expected_samples and len(valid_window) == expected_samples

    if valid_window.empty:
        min_value = float('nan')
        max_value = float('nan')
        outside_limit_count = 0
    else:
        min_value = valid_window.min()
        max_value = valid_window.max()
        outside_limit_count = int(
            (~valid_window.between(lower_limit, upper_limit, inclusive='both')).sum()
        )

    within_limits = bool(full_coverage and outside_limit_count == 0)

    return pd.Series(
        {
            'expected_samples': expected_samples,
            'observed_samples': observed_samples,
            'full_coverage': full_coverage,
            'outside_limit_count': outside_limit_count,
            'min_fi1000': min_value,
            'max_fi1000': max_value,
            'within_limits': within_limits,
        }
    )


summary_rows = []
fi1000_alarm_pass_tables = {}

print(
    'Envelope expansion rule: add the stated percentage of the base FI_1000 span to both sides of the base limits.'
 )
print(f'Total rows in overlapping_alarms: {raw_alarm_count}')
print(f'Alarm windows with valid start/end timestamps: {len(alarm_windows)}')

for expansion_fraction in expansion_schedule:
    expansion_pct = int(expansion_fraction * 100)
    margin = base_span * expansion_fraction
    lower_limit = LOWER_LIMIT_FI1000 - margin
    upper_limit = UPPER_LIMIT_FI1000 + margin

    scenario_eval = alarm_windows.apply(
        lambda row: evaluate_alarm_window(
            row['sample_start'],
            row['sample_end'],
            lower_limit,
            upper_limit,
        ),
        axis=1,
    )

    scenario_results = pd.concat(
        [
            alarm_windows[['alarm_id', 'alarm_start', 'alarm_end', 'sample_start', 'sample_end']],
            scenario_eval,
        ],
        axis=1,
    )
    
    passing_alarms = scenario_results.loc[scenario_results['within_limits']].copy()
    passing_alarms['min_fi1000'] = passing_alarms['min_fi1000'].round(6)
    passing_alarms['max_fi1000'] = passing_alarms['max_fi1000'].round(6)
    fi1000_alarm_pass_tables[expansion_pct] = passing_alarms[
        ['alarm_id', 'alarm_start', 'alarm_end', 'min_fi1000', 'max_fi1000', 'expected_samples']
    ].reset_index(drop=True)

    passing_alarm_count = len(passing_alarms)
    total_alarm_count = len(scenario_results)
    full_coverage_alarm_count = int(scenario_results['full_coverage'].sum())

    summary_rows.append(
        {
            'scenario': 'base_limits' if expansion_pct == 0 else f'base_plus_{expansion_pct}pct',
            'expansion_pct': expansion_pct,
            'lower_limit': round(lower_limit, 6),
            'upper_limit': round(upper_limit, 6),
            'passing_alarm_count': passing_alarm_count,
            'total_alarm_count': total_alarm_count,
            'passing_alarm_pct': round(100 * passing_alarm_count / total_alarm_count, 2),
            'full_coverage_alarm_count': full_coverage_alarm_count,
            'not_full_coverage_alarm_count': total_alarm_count - full_coverage_alarm_count,
        }
    )

fi1000_alarm_envelope_summary = pd.DataFrame(summary_rows)
display(fi1000_alarm_envelope_summary)

for expansion_pct in [0, 10, 20, 30]:
    passing_alarms = fi1000_alarm_pass_tables[expansion_pct]
    scenario_label = 'Base limits' if expansion_pct == 0 else f'Base limits + {expansion_pct}% expansion'
    print(
        f'\n{scenario_label}: {len(passing_alarms)} alarms stay within the evaluated FI_1000 limits for the full alarm duration.'
    )
    display(passing_alarms)

Envelope expansion rule: add the stated percentage of the base FI_1000 span to both sides of the base limits.
Total rows in overlapping_alarms: 539
Alarm windows with valid start/end timestamps: 539


,scenario,expansion_pct,lower_limit,upper_limit,passing_alarm_count,total_alarm_count,passing_alarm_pct,full_coverage_alarm_count,not_full_coverage_alarm_count
0,base_limits,0,8.386506,8.630340,57,539,10.58,528,11
1,base_plus_10pct,10,8.362122,8.654723,80,539,14.84,528,11
2,base_plus_20pct,20,8.337739,8.679107,103,539,19.11,528,11
3,base_plus_30pct,30,8.313356,8.703490,122,539,22.63,528,11



Base limits: 57 alarms stay within the evaluated FI_1000 limits for the full alarm duration.


,alarm_id,alarm_start,alarm_end,min_fi1000,max_fi1000,expected_samples
0,171,2023-05-02 04:42:34.805,2023-05-02 04:49:03.810,8.547328,8.611371,7
1,200,2023-08-09 13:26:54.004,2023-08-09 13:33:46.004,8.418064,8.496234,7
2,201,2023-08-10 16:58:00.004,2023-08-10 17:05:25.004,8.491291,8.620660,7
3,209,2023-10-25 01:45:14.104,2023-10-25 01:58:30.353,8.591278,8.591731,13
4,210,2023-10-25 02:46:07.104,2023-10-25 02:55:15.102,8.589126,8.589428,9
5,211,2023-10-25 03:56:16.104,2023-10-25 04:11:40.102,8.586257,8.586785,15
6,218,2024-01-02 05:50:15.102,2024-01-02 06:10:40.154,8.408764,8.603221,20
7,226,2024-01-05 11:01:11.952,2024-01-05 11:07:29.953,8.388277,8.490507,6
8,228,2024-01-05 17:21:03.352,2024-01-05 17:24:44.357,8.411674,8.461634,3
9,230,2024-01-06 02:17:57.203,2024-01-06 02:22:53.201,8.443940,8.458315,5



Base limits + 10% expansion: 80 alarms stay within the evaluated FI_1000 limits for the full alarm duration.


,alarm_id,alarm_start,alarm_end,min_fi1000,max_fi1000,expected_samples
0,171,2023-05-02 04:42:34.805,2023-05-02 04:49:03.810,8.547328,8.611371,7
1,172,2023-05-02 06:11:50.852,2023-05-02 06:18:34.852,8.482380,8.637196,7
2,176,2023-05-21 00:05:56.808,2023-05-21 00:11:22.804,8.567216,8.632663,6
3,200,2023-08-09 13:26:54.004,2023-08-09 13:33:46.004,8.418064,8.496234,7
4,201,2023-08-10 16:58:00.004,2023-08-10 17:05:25.004,8.491291,8.620660,7
...,...,...,...,...,...,...
75,492,2025-04-20 07:29:40.425,2025-04-20 07:35:44.424,8.392848,8.540930,6
76,515,2025-05-21 14:17:53.579,2025-05-21 14:18:37.723,8.553716,8.553716,1
77,528,2025-06-17 17:30:50.107,2025-06-17 17:31:59.223,8.540364,8.540364,1
78,532,2025-06-19 12:31:42.504,2025-06-19 12:32:23.198,8.397397,8.397397,1



Base limits + 20% expansion: 103 alarms stay within the evaluated FI_1000 limits for the full alarm duration.


,alarm_id,alarm_start,alarm_end,min_fi1000,max_fi1000,expected_samples
0,171,2023-05-02 04:42:34.805,2023-05-02 04:49:03.810,8.547328,8.611371,7
1,172,2023-05-02 06:11:50.852,2023-05-02 06:18:34.852,8.482380,8.637196,7
2,176,2023-05-21 00:05:56.808,2023-05-21 00:11:22.804,8.567216,8.632663,6
3,200,2023-08-09 13:26:54.004,2023-08-09 13:33:46.004,8.418064,8.496234,7
4,201,2023-08-10 16:58:00.004,2023-08-10 17:05:25.004,8.491291,8.620660,7
...,...,...,...,...,...,...
98,515,2025-05-21 14:17:53.579,2025-05-21 14:18:37.723,8.553716,8.553716,1
99,517,2025-05-22 12:13:57.733,2025-05-22 12:15:25.990,8.621990,8.674137,2
100,528,2025-06-17 17:30:50.107,2025-06-17 17:31:59.223,8.540364,8.540364,1
101,532,2025-06-19 12:31:42.504,2025-06-19 12:32:23.198,8.397397,8.397397,1



Base limits + 30% expansion: 122 alarms stay within the evaluated FI_1000 limits for the full alarm duration.


,alarm_id,alarm_start,alarm_end,min_fi1000,max_fi1000,expected_samples
0,116,2022-12-08 07:59:18.355,2022-12-08 08:06:23.353,8.328461,8.422258,7
1,171,2023-05-02 04:42:34.805,2023-05-02 04:49:03.810,8.547328,8.611371,7
2,172,2023-05-02 06:11:50.852,2023-05-02 06:18:34.852,8.482380,8.637196,7
3,176,2023-05-21 00:05:56.808,2023-05-21 00:11:22.804,8.567216,8.632663,6
4,178,2023-06-04 14:05:18.855,2023-06-04 14:15:02.406,8.333061,8.495438,10
...,...,...,...,...,...,...
117,517,2025-05-22 12:13:57.733,2025-05-22 12:15:25.990,8.621990,8.674137,2
118,528,2025-06-17 17:30:50.107,2025-06-17 17:31:59.223,8.540364,8.540364,1
119,531,2025-06-18 16:53:59.501,2025-06-18 16:54:47.382,8.336717,8.336717,1
120,532,2025-06-19 12:31:42.504,2025-06-19 12:32:23.198,8.397397,8.397397,1


In [9]:
# Most operated tags for alarms within the FI_1000 30% expansion band
import pandas as pd
from IPython.display import display

target_expansion_fraction = 0.30
target_expansion_pct = int(target_expansion_fraction * 100)
target_margin = base_span * target_expansion_fraction
target_lower_limit = LOWER_LIMIT_FI1000 - target_margin
target_upper_limit = UPPER_LIMIT_FI1000 + target_margin

fi1000_30pct_eval = alarm_windows.apply(
    lambda row: evaluate_alarm_window(
        row['sample_start'],
        row['sample_end'],
        target_lower_limit,
        target_upper_limit,
    ),
    axis=1,
 )

fi1000_30pct_results = pd.concat(
    [
        alarm_windows[['alarm_id', 'alarm_start', 'alarm_end', 'sample_start', 'sample_end']],
        fi1000_30pct_eval,
    ],
    axis=1,
 )

fi1000_30pct_passing_alarms = fi1000_30pct_results.loc[
    fi1000_30pct_results['within_limits']
].copy()
fi1000_30pct_passing_alarms['min_fi1000'] = fi1000_30pct_passing_alarms['min_fi1000'].round(6)
fi1000_30pct_passing_alarms['max_fi1000'] = fi1000_30pct_passing_alarms['max_fi1000'].round(6)

if 'control_actions' not in globals():
    control_actions = pd.read_excel(
        alarm_workbook_path,
        sheet_name='control_actions',
    )

filtered_control_actions = control_actions.copy()
for col in ['cluster_start', 'cluster_end', 'VT_Start']:
    filtered_control_actions[col] = pd.to_datetime(filtered_control_actions[col], errors='coerce')
    filtered_control_actions[col] = strip_timezone(filtered_control_actions[col])

qualified_alarm_lookup = fi1000_30pct_passing_alarms[
    ['alarm_id', 'alarm_start', 'alarm_end']
].rename(columns={'alarm_start': 'cluster_start', 'alarm_end': 'cluster_end'})

qualified_alarm_actions = filtered_control_actions.merge(
    qualified_alarm_lookup,
    on=['cluster_start', 'cluster_end'],
    how='inner',
 )

qualified_alarm_actions = qualified_alarm_actions.loc[
    qualified_alarm_actions['Description'].isin(['OP', 'SP'])
].copy()

if qualified_alarm_actions.empty:
    raise ValueError(
        'No OP/SP control-action rows were matched to the alarms inside the FI_1000 30% expansion band.'
    )

action_count_by_tag = (
    qualified_alarm_actions.groupby(['Description', 'Source'])
    .agg(
        action_count=('Source', 'size'),
        unique_alarm_count=('alarm_id', 'nunique'),
    )
    .reset_index()
    .rename(columns={'Description': 'action_type', 'Source': 'tag'})
 )

timing_breakdown = (
    qualified_alarm_actions.groupby(['Description', 'Source', 'action_timing'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
    .rename(columns={'Description': 'action_type', 'Source': 'tag'})
 )

tag_operation_summary = action_count_by_tag.merge(
    timing_breakdown,
    on=['action_type', 'tag'],
    how='left',
 )

sort_cols = ['action_count', 'unique_alarm_count', 'tag']
op_most_operated_tags = (
    tag_operation_summary.loc[tag_operation_summary['action_type'] == 'OP']
    .sort_values(sort_cols, ascending=[False, False, True])
    .reset_index(drop=True)
 )
sp_most_operated_tags = (
    tag_operation_summary.loc[tag_operation_summary['action_type'] == 'SP']
    .sort_values(sort_cols, ascending=[False, False, True])
    .reset_index(drop=True)
 )

print(
    f'FI_1000 30% expansion band: [{target_lower_limit:.6f}, {target_upper_limit:.6f}]'
 )
print(
    f'{len(fi1000_30pct_passing_alarms)} alarms stay inside this band for the full alarm duration.'
 )
print(
    f"{qualified_alarm_actions['alarm_id'].nunique()} of those alarms have matched OP/SP control-action rows, totaling {len(qualified_alarm_actions)} action rows."
 )

display(
    fi1000_30pct_passing_alarms[
        ['alarm_id', 'alarm_start', 'alarm_end', 'min_fi1000', 'max_fi1000', 'expected_samples']
    ].reset_index(drop=True)
 )

print('\nMost operated tags - OP actions')
display(op_most_operated_tags)

print('\nMost operated tags - SP actions')
display(sp_most_operated_tags)

FI_1000 30% expansion band: [8.313356, 8.703490]
122 alarms stay inside this band for the full alarm duration.
74 of those alarms have matched OP/SP control-action rows, totaling 995 action rows.


,alarm_id,alarm_start,alarm_end,min_fi1000,max_fi1000,expected_samples
0,116,2022-12-08 07:59:18.355,2022-12-08 08:06:23.353,8.328461,8.422258,7
1,171,2023-05-02 04:42:34.805,2023-05-02 04:49:03.810,8.547328,8.611371,7
2,172,2023-05-02 06:11:50.852,2023-05-02 06:18:34.852,8.482380,8.637196,7
3,176,2023-05-21 00:05:56.808,2023-05-21 00:11:22.804,8.567216,8.632663,6
4,178,2023-06-04 14:05:18.855,2023-06-04 14:15:02.406,8.333061,8.495438,10
...,...,...,...,...,...,...
117,517,2025-05-22 12:13:57.733,2025-05-22 12:15:25.990,8.621990,8.674137,2
118,528,2025-06-17 17:30:50.107,2025-06-17 17:31:59.223,8.540364,8.540364,1
119,531,2025-06-18 16:53:59.501,2025-06-18 16:54:47.382,8.336717,8.336717,1
120,532,2025-06-19 12:31:42.504,2025-06-19 12:32:23.198,8.397397,8.397397,1



Most operated tags - OP actions


,action_type,tag,action_count,unique_alarm_count,after,before,during
0,OP,03LIC_1071,294,11,92,52,150
1,OP,03PIC_1013,101,22,21,64,16
2,OP,03FIC_3435,101,16,29,61,11
3,OP,03HIC_1151,42,6,15,21,6
4,OP,03HIC_1141,37,3,15,12,10
5,OP,03HIC_1183,31,3,9,14,8
6,OP,03FIC_1085,27,1,0,27,0
7,OP,03FIC_3415,22,3,2,14,6
8,OP,03LIC_1094,20,1,20,0,0
9,OP,03XAX_1002,14,2,0,4,10



Most operated tags - SP actions


,action_type,tag,action_count,unique_alarm_count,after,before,during
0,SP,03LIC_1034,145,26,34,78,33
1,SP,03LIC_1071,57,14,37,3,17
2,SP,03TIC_1009,22,10,6,15,1
3,SP,03LIC_1085,20,8,7,8,5
4,SP,03LIC_1016,11,6,3,2,6
5,SP,03PIC_1068,5,2,4,0,1
6,SP,03LIC_3408,2,2,1,1,0
7,SP,03PIC_3131,2,1,0,0,2
8,SP,03LIC_1031,1,1,0,1,0
9,SP,03LIC_1097,1,1,1,0,0


In [10]:
# Alarms within the FI_1000 30% expansion band that have no matched OP/SP control actions
matched_op_sp_alarm_ids = set(qualified_alarm_actions['alarm_id'].unique())

fi1000_30pct_alarms_without_op_sp_actions = (
    fi1000_30pct_passing_alarms.loc[
        ~fi1000_30pct_passing_alarms['alarm_id'].isin(matched_op_sp_alarm_ids),
        ['alarm_id', 'alarm_start', 'alarm_end', 'min_fi1000', 'max_fi1000', 'expected_samples'],
    ]
    .sort_values('alarm_id')
    .reset_index(drop=True)
 )

print(
    f"{len(fi1000_30pct_alarms_without_op_sp_actions)} alarms inside the FI_1000 30% expansion band have no matched OP/SP control-action rows."
 )
display(fi1000_30pct_alarms_without_op_sp_actions)

48 alarms inside the FI_1000 30% expansion band have no matched OP/SP control-action rows.


,alarm_id,alarm_start,alarm_end,min_fi1000,max_fi1000,expected_samples
0,116,2022-12-08 07:59:18.355,2022-12-08 08:06:23.353,8.328461,8.422258,7
1,171,2023-05-02 04:42:34.805,2023-05-02 04:49:03.810,8.547328,8.611371,7
2,200,2023-08-09 13:26:54.004,2023-08-09 13:33:46.004,8.418064,8.496234,7
3,226,2024-01-05 11:01:11.952,2024-01-05 11:07:29.953,8.388277,8.490507,6
4,228,2024-01-05 17:21:03.352,2024-01-05 17:24:44.357,8.411674,8.461634,3
5,229,2024-01-05 22:36:21.307,2024-01-05 22:40:05.104,8.597386,8.665996,4
6,230,2024-01-06 02:17:57.203,2024-01-06 02:22:53.201,8.443940,8.458315,5
7,234,2024-01-06 11:01:40.353,2024-01-06 11:05:59.354,8.489911,8.518058,4
8,235,2024-01-06 17:59:45.404,2024-01-06 18:04:41.404,8.535021,8.583914,5
9,241,2024-01-07 04:02:09.352,2024-01-07 04:05:14.354,8.471346,8.509773,3


In [11]:
# Alarm IDs only: FI_1000 30% expansion band alarms without matched OP/SP control actions
no_op_sp_alarm_ids = fi1000_30pct_alarms_without_op_sp_actions['alarm_id'].tolist()
print(no_op_sp_alarm_ids)

[116, 171, 200, 226, 228, 229, 230, 234, 235, 241, 242, 244, 245, 253, 290, 295, 297, 303, 306, 307, 308, 318, 319, 320, 321, 322, 326, 329, 331, 382, 394, 413, 415, 419, 420, 422, 427, 430, 431, 435, 464, 477, 478, 479, 482, 487, 490, 492]
